In [1]:

"""
@utor: Yuri Idalgo de Matos da Silva
criação: 15/10/2026
atualização: 15/10/2026 
descrição:
"""

'\n@utor: Yuri Idalgo de Matos da Silva\ncriação: 15/10/2026\natualização: 15/10/2026 \ndescrição:\n'

Add. Bibliotecas e scripts necessários para geocodificação.

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


In [3]:
# Leitura de dados

file_path = Path(
    r"C:\Users\x432601\Documents\GitHub\ETL_wifi_livre_2026\data_manual"
    ) 
path = file_path / "pontos_programa_wifi-livre-sp.txt"

df = pd.read_csv(path, sep=";", encoding="utf-8")

In [4]:
# Conferindo os dados
df.head()

,Tipo de Localidade,Logradouro,Bairro,CEP,Região,Pontos de Acesso
0,Abrigo de Ônibus,"Avenida Treze de Maio, 1434",Bela Vista,01321-010,Centro,1
1,Abrigo de Ônibus,"Avenida Rudge, 810",Bom Retiro,01536-000,Centro,1
2,Abrigo de Ônibus,"Rua Conselheiro Furtado, 1246",Liberdade,01007-040,Centro,1
3,Abrigo de Ônibus,"Avenida Mario de Andrade, 48",Santa Cecília,01153-000,Centro,1
4,Abrigo de Ônibus,"Avenida Conselheiro Carrão, 1706",Carrão,01020-030,Leste,1


In [5]:
#Add. cidade e estado

df['Cidade'] = 'São Paulo'
df['Estado'] = 'SP'
df['Pais'] = 'Brasil'

In [6]:
# Verifição dataframe
df.head()

,Tipo de Localidade,Logradouro,Bairro,CEP,Região,Pontos de Acesso,Cidade,Estado,Pais
0,Abrigo de Ônibus,"Avenida Treze de Maio, 1434",Bela Vista,01321-010,Centro,1,São Paulo,SP,Brasil
1,Abrigo de Ônibus,"Avenida Rudge, 810",Bom Retiro,01536-000,Centro,1,São Paulo,SP,Brasil
2,Abrigo de Ônibus,"Rua Conselheiro Furtado, 1246",Liberdade,01007-040,Centro,1,São Paulo,SP,Brasil
3,Abrigo de Ônibus,"Avenida Mario de Andrade, 48",Santa Cecília,01153-000,Centro,1,São Paulo,SP,Brasil
4,Abrigo de Ônibus,"Avenida Conselheiro Carrão, 1706",Carrão,01020-030,Leste,1,São Paulo,SP,Brasil


In [7]:
# Combinações de parâmetros para o geocoder
df['localizacao_standard'] = df['Logradouro'] + ', ' + df['Cidade'] + '-' + df['Estado'] + ', '+ df['CEP'] + ', ' + df['Pais']
df['localizacao_standard'].head()

0    Avenida Treze de Maio, 1434, São Paulo-SP, 013...
1    Avenida Rudge, 810, São Paulo-SP, 01536-000, B...
2    Rua Conselheiro Furtado, 1246, São Paulo-SP, 0...
3    Avenida Mario de Andrade, 48, São Paulo-SP, 01...
4    Avenida Conselheiro Carrão, 1706, São Paulo-SP...
Name: localizacao_standard, dtype: str

In [8]:
# Print de verificação
print(df['localizacao_standard'].to_list()[0])
print(df['localizacao_standard'].shape)
print(df['localizacao_standard'].unique().shape)

# Remoção de valores duplicados com base na coluna 'localizacao_standard'
df = df.drop_duplicates(subset=['localizacao_standard'])

Avenida Treze de Maio, 1434, São Paulo-SP, 01321-010, Brasil
(2945,)
(2465,)


In [9]:

geolocator = Nominatim(user_agent="wifi_livre_2026")

# Garante delay mínimo de 1 segundo entre chamadas e 2 tentativas em caso de erro
geocode_com_delay = RateLimiter(
    geolocator.geocode, 
    min_delay_seconds=1.0, 
    max_retries=2, 
    error_wait_seconds=5.0
)

# Aplica a busca criando a coluna com o objeto retornado
df["geolocalizacao"] = df["localizacao_standard"].apply(geocode_com_delay)

# Extrai coordenadas numéricas
df["latitude"] = df["geolocalizacao"].apply(lambda loc: loc.latitude if loc else None)
df["longitude"] = df["geolocalizacao"].apply(lambda loc: loc.longitude if loc else None)

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Rua Doutor Rafael de Barros, 132, São Paulo-SP, 03215-080, Brasil',), **{}).
Traceback (most recent call last):
  File "c:\Users\x432601\AppData\Local\anaconda3\envs\wifi_livre_sp\Lib\site-packages\urllib3\connectionpool.py", line 540, in _make_request
    response = conn.getresponse()
  File "c:\Users\x432601\AppData\Local\anaconda3\envs\wifi_livre_sp\Lib\site-packages\urllib3\connection.py", line 638, in getresponse
    httplib_response = super().getresponse()
  File "c:\Users\x432601\AppData\Local\anaconda3\envs\wifi_livre_sp\Lib\http\client.py", line 1478, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "c:\Users\x432601\AppData\Local\anaconda3\envs\wifi_livre_sp\Lib\http\client.py", line 343, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\x432601\AppData\Local\anaconda3\envs\wifi_livre_sp\Lib\http\client.py", line 304, in

In [10]:
# Salvando dados processados.
df.to_csv(file_path + "pontos_programa_wifi-livre-sp_geolocalizado.csv", sep=";", index=False, encoding="utf-8")

TypeError: unsupported operand type(s) for +: 'WindowsPath' and 'str'

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print(df.columns.to_list())

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path(r"C:\Users\x432601\Documents\GitHub\ETL_wifi_livre_2026\data_manual")
path = file_path / "pontos_programa_wifi-livre-sp_geolocalizado.csv"

df = pd.read_csv(path, sep=";", encoding="utf-8")

In [ ]:
# Transfomando para GeoJSON
# 1. Filtra as linhas com coordenadas válidas
df_validos = df.dropna(subset=['latitude', 'longitude']).copy()

# 2. Converte todos os NaN / nulos para string vazia "" ou None
df_validos = df_validos.fillna("")  # ou df_validos.replace({np.nan: None})

# 3. Remoção de valores duplicados com base na coluna 'localizacao_standard'
df_validos = df_validos.drop_duplicates(subset=['localizacao_standard'])

# 4. Criação do GeoJSON

features = []

for _, row in df_validos.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row['longitude']), float(row['latitude'])]
        },
        "properties": {
            "tipo_localidade": row.get('Tipo de Localidade'),
            "logradouro": row.get('Logradouro'),
            "bairro": row.get('Bairro'),
            "cep": str(row.get('CEP')),
            "regiao": row.get('Região'),
            "pontos_acesso": row.get('Pontos de Acesso '),
            "cidade": row.get('Cidade'),
            "estado": row.get('Estado'),
            "pais": row.get('Pais'),
            "localizacao_standard": row.get('localizacao_standard')
        }
    }
    features.append(feature)

geojson_data = {
    "type": "FeatureCollection",
    "features": features
}

# 3. Guarda novamente o ficheiro corrigido
with open('pontos_wifi.geojson', 'w', encoding='utf-8') as f:
    json.dump(geojson_data, f, ensure_ascii=False, indent=2)

print("GeoJSON gerado sem NaN!")

In [ ]:
df_validos.info()

In [ ]:
df_validos.shape

In [ ]:
# Observação: A coluna 'localizacao_standard' é usada para identificar registros únicos de localização e parece que tá contendo valores duplicados. Para garantir que cada ponto de acesso seja representado apenas uma vez no GeoJSON, é importante verificar a quantidade de valores únicos nessa coluna.
print(len(df_validos['localizacao_standard'].unique()))